## Entropy
**Entropy** measures the disorder or uncertainty in a system. The calculation depends on context—here are the main types:

### Information Entropy (Shannon Entropy)

For a discrete probability distribution:

**H(X) = -Σ p(x) × log₂(p(x))**

Where:
- p(x) = probability of outcome x
- log₂ = logarithm base 2 (gives entropy in bits)
- The sum is over all possible outcomes

**Example:** Fair coin flip
- P(heads) = 0.5, P(tails) = 0.5
- H = -(0.5 × log₂(0.5) + 0.5 × log₂(0.5))
- H = -(0.5 × -1 + 0.5 × -1) = 1 bit

**Example:** Biased coin (80% heads, 20% tails)
- H = -(0.8 × log₂(0.8) + 0.2 × log₂(0.2))
- H ≈ 0.72 bits (less uncertainty)

### Cross-Entropy (Machine Learning)

Used to measure difference between predicted and actual probability distributions:

**H(P, Q) = -Σ p(x) × log(q(x))**

Where p(x) is the true distribution and q(x) is the predicted distribution.

### Key Points

- **High entropy** = high uncertainty/disorder (uniform distribution)
- **Low entropy** = low uncertainty/predictability (concentrated distribution)
- Natural logarithm (ln) gives entropy in nats; log base 2 gives bits
- Entropy is always ≥ 0

**For machine learning**, cross-entropy is commonly used as a loss function in classification tasks.

#
---
---

## Cross-Entropy

Cross-entropy measures how well a predicted probability distribution matches the true distribution. It comes from information theory.

**Formula**

For a true distribution *p* and predicted distribution *q*:

$$H(p, q) = -\sum_x p(x) \log q(x)$$

In classification with a one-hot true label (class *c* is correct):

$$H = -\log q(c)$$

So it's just the negative log of the probability assigned to the correct class.

**Intuition**

- If your model assigns probability 1.0 to the correct class → loss = 0
- If it assigns 0.5 → loss = 0.69
- If it assigns 0.01 → loss = 4.6

The loss penalizes overconfidence in wrong answers *exponentially*.

**Relation to KL Divergence**

$$H(p, q) = H(p) + D_{KL}(p \| q)$$

Cross-entropy = entropy of the true distribution + KL divergence. Minimizing cross-entropy is equivalent to minimizing KL divergence (since *H(p)* is fixed w.r.t. model parameters).

**Why it's used in ML**

- It's the natural loss for maximum likelihood estimation with softmax outputs
- Gradients are well-behaved (no vanishing gradient problem that MSE has for classification)
- For binary classification it simplifies to: $-[y \log \hat{y} + (1-y)\log(1-\hat{y})]$

**Binary vs. Categorical**

| | Binary | Categorical |
|---|---|---|
| Classes | 2 | K |
| Output | sigmoid | softmax |
| Loss | Binary CE | Categorical CE |

Is there a specific aspect you want to dig into — the math, the gradient derivation, or how it's used in a particular context like LLMs or neural nets?
#
---
---

In the formula:

$$H(p, q) = H(p) + D_{KL}(p \| q)$$

*H(p)* is the entropy of the **true label distribution** — it depends only on your dataset, not on your model parameters.

**Concrete example:** in a classification task, the true label for a given sample is fixed (e.g., "this image is a cat"). That's determined by your data. Your model parameters *θ* only affect *q* (the predicted probabilities), never *p*.

So when you differentiate the loss w.r.t. *θ*:

$$\frac{\partial}{\partial \theta} H(p, q) = \frac{\partial}{\partial \theta} H(p) + \frac{\partial}{\partial \theta} D_{KL}(p \| q)$$

The first term is zero — *H(p)* has no *θ* in it. Therefore minimizing cross-entropy and minimizing KL divergence are identical optimization problems.

**The intuition:** you can't change how uncertain the ground truth is. You can only change how well your predictions match it. Cross-entropy measures both the irreducible uncertainty in the data (*H(p)*) plus the extra "mistake" your model makes (*D_KL*). Since you can't reduce the first term, minimizing cross-entropy is entirely about minimizing that gap.

#
---
---

**Entropy** $H(p)$ measures the **uncertainty in a single distribution**:

$$H(p) = -\sum_x p(x) \log p(x)$$

It asks: *"how unpredictable is p itself?"*

- Fair coin (50/50) → high entropy
- Biased coin (99/1) → low entropy
- No model involved — just one distribution

---

**Cross-entropy** $H(p, q)$ measures **how well q approximates p**:

$$H(p, q) = -\sum_x p(x) \log q(x)$$

It asks: *"how many bits do I need if I encode p using q's codebook?"*

- Two distributions involved — truth p and prediction q
- Always ≥ H(p)
- Equals H(p) only when q = p exactly

---

**Side by side**

| | Entropy | Cross-Entropy |
|---|---|---|
| Formula | $-\sum p \log p$ | $-\sum p \log q$ |
| Distributions | One (p) | Two (p and q) |
| Measures | Intrinsic uncertainty of p | How well q approximates p |
| Minimum | 0 (deterministic) | H(p) (when q = p) |
| Used for | Analyzing data | Training ML models |

---

**The relationship**

$$H(p, q) = H(p) + D_{KL}(p \| q)$$

Cross-entropy = unavoidable uncertainty + extra cost from using the wrong model. Entropy is just the lower bound you can never beat — the irreducible uncertainty baked into your data.

**One-line summary:** entropy measures how uncertain the truth is; cross-entropy measures how wrong your model is about the truth.

#
---
---

$H(p)$ is the entropy of the true distribution $p$.

The notation just makes explicit *which* distribution's entropy you're computing:

- $H(p)$ — entropy of $p$
- $H(q)$ — entropy of $q$  
- $H(p, q)$ — cross-entropy *between* $p$ and $q$

So when you see the decomposition:

$$H(p, q) = H(p) + D_{KL}(p \| q)$$

it's saying: cross-entropy = **entropy of the true distribution** + **extra cost from your model's mistakes**.

The comma vs. single argument is the only syntactic difference between cross-entropy and entropy. Easy to miss but important.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

# Load & split
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

# Predict probabilities
y_prob = model.predict_proba(X_test)

# Cross-entropy per sample: -log(P(true class))
sample_losses = [-np.log(y_prob[i, y_test[i]]) for i in range(len(y_test))]

# Print results
print(f"Accuracy         : {model.score(X_test, y_test):.2f}")
print(f"Cross-Entropy    : {log_loss(y_test, y_prob):.4f}") # log_loss(y_test, y_prob) is the shortcut for -np.log(P(true label))
print(f"Manual CE (avg)  : {np.mean(sample_losses):.4f}")

Accuracy         : 1.00
Cross-Entropy    : 0.1152
Manual CE (avg)  : 0.1152
